# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: TÙY CHỌN khi thực thi mã C++</h2>
            <span style="color:#f71;">Cách khác: bạn có thể chạy trên website đã giới thiệu hôm qua</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model mã nguồn mở miễn phí trên Ollama. Mình cũng dùng các model mã nguồn mở trả phí qua Groq và OpenRouter. Chỉ chọn những model bạn muốn!
            </span>
        </td>
    </tr>
</table>

In [33]:
# Ô này nạp các thư viện cần dùng cho notebook Day 4 (thêm Gradio so với Day 3).

import os  # Đọc biến môi trường, ví dụ API Key.
import io  # StringIO: bộ đệm giả stdout để bắt print() đưa lên Gradio.
import sys  # Đổi tạm sys.stdout khi bắt output.
import shutil  # Tìm compiler C++ trong PATH.
from dotenv import load_dotenv  # Nạp khóa từ file .env.
from openai import OpenAI  # Client Chat Completions; dùng cho nhiều nhà cung cấp nhờ base_url.
import gradio as gr  # UI web: ô code, dropdown model, nút chuyển mã.
import subprocess  # Chạy compiler và file thực thi từ Python.

In [19]:
load_dotenv(override=True)  # Đọc .env; override=True ghi đè biến môi trường cũ.

openai_api_key = os.getenv('OPENAI_API_KEY')  # Khóa OpenAI.
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')  # Khóa Claude (tùy chọn).
google_api_key = os.getenv('GOOGLE_API_KEY')  # Khóa Gemini (tùy chọn).
grok_api_key = os.getenv('GROK_API_KEY')  # Khóa Grok (tùy chọn).
groq_api_key = os.getenv('GROQ_API_KEY')  # Khóa Groq — chạy model OSS trên cloud (tùy chọn).
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')  # Khóa OpenRouter — cổng nhiều model (tùy chọn).

if openai_api_key:  # Có khóa: in vài ký tự đầu để xác nhận, không lộ hết khóa.
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")  # Chỉ in phần đầu để xác nhận, tránh lộ toàn bộ khóa.
else:  # Chạy khi không tìm thấy khóa OpenAI.
    print("Chưa thiết lập OpenAI API Key")  # Thông báo OpenAI chưa được cấu hình.

if anthropic_api_key:  # Kiểm tra khóa Anthropic có tồn tại hay không.
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")  # Chỉ in 7 ký tự đầu để xác nhận.
else:  # Chạy khi chưa có khóa Anthropic.
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")  # Nhắc rằng Claude là lựa chọn tùy chọn.

if google_api_key:  # Kiểm tra khóa Google có tồn tại hay không.
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")  # Chỉ in 2 ký tự đầu để nhận biết khóa.
else:  # Chạy khi chưa có khóa Google.
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")  # Báo rằng Gemini chưa được cấu hình.

if grok_api_key:  # Kiểm tra khóa Grok có tồn tại hay không.
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")  # Chỉ in 4 ký tự đầu của khóa.
else:  # Chạy khi chưa có khóa Grok.
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")  # Báo rằng Grok chưa được cấu hình.

if groq_api_key:  # Kiểm tra khóa Groq có tồn tại hay không.
    print(f"Groq API Key tồn tại và bắt đầu bằng {groq_api_key[:4]}")  # Chỉ in phần đầu khóa để xác nhận.
else:  # Chạy khi chưa có khóa Groq.
    print("Chưa thiết lập Groq API Key (và đây là tùy chọn)")  # Báo rằng Groq chưa được cấu hình.

if openrouter_api_key:  # Kiểm tra khóa OpenRouter có tồn tại hay không.
    print(f"OpenRouter API Key tồn tại và bắt đầu bằng {openrouter_api_key[:6]}")  # Chỉ in 6 ký tự đầu để xác nhận.
else:  # Chạy khi chưa có khóa OpenRouter.
    print("Chưa thiết lập OpenRouter API Key (và đây là tùy chọn)")  # Báo rằng OpenRouter chưa được cấu hình.



OpenAI API Key tồn tại và bắt đầu bằng sk-proj-
Anthropic API Key tồn tại và bắt đầu bằng sk-ant-
Google API Key tồn tại và bắt đầu bằng AQ
Chưa thiết lập Grok API Key (và đây là tùy chọn)
Groq API Key tồn tại và bắt đầu bằng gsk_
OpenRouter API Key tồn tại và bắt đầu bằng sk-or-


In [20]:
# Tạo client: cùng class OpenAI, khác base_url và khóa.

openai = OpenAI()  # OpenAI mặc định (api.openai.com).

anthropic_url = "https://api.anthropic.com/v1/"  # Claude.
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini.
grok_url = "https://api.x.ai/v1"  # Grok.
groq_url = "https://api.groq.com/openai/v1"  # Groq (inference nhanh trên cloud).
ollama_url = "http://localhost:11434/v1"  # Ollama local (máy bạn phải đang chạy ollama).
openrouter_url = "https://openrouter.ai/api/v1"  # OpenRouter.

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)  # Client Claude.
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)  # Client Gemini.
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)  # Client Grok.
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)  # Client Groq.
ollama = OpenAI(api_key="ollama", base_url=ollama_url)  # Ollama không cần khóa thật; chuỗi "ollama" chỉ để SDK không báo thiếu key.
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)  # Client OpenRouter.


In [21]:
# Danh sách tên model hiện trên dropdown Gradio (thứ tự = thứ tự trong menu).
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-3.1-pro-preview", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b"]  # Các model cloud/local mà người dùng có thể chọn.

# Map tên model -> đúng client (cùng SDK, khác endpoint).
clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-3.1-pro-preview": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}  # Cho phép hàm port chọn đúng endpoint từ tên model.

# Muốn giữ chi phí cực thấp? Thay list models bằng model rẻ hơn, giống ví dụ Day 3.

In [22]:
from system_info import retrieve_system_info  # Nhập hàm thu thập cấu hình hệ thống của máy.

system_info = retrieve_system_info()  # Lấy hệ điều hành, CPU và môi trường để đưa vào prompt tối ưu mã.
system_info  # Hiển thị thông tin hệ thống trong output của ô notebook.

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i9-9900K CPU @ 3.60GHz',
  'cores_logical': 16,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Ghi đè các lệnh này bằng lệnh từ hôm qua

Hoặc dùng website như hôm qua:

 https://www.programiz.com/cpp-programming/online-compiler/

In [34]:
# Tìm compiler C++ có sẵn trong PATH và chọn lệnh chạy phù hợp với hệ điều hành.
compiler = shutil.which("clang++") or shutil.which("g++") or shutil.which("cl")
if compiler is None:
    compile_command = None
    run_command = None
    print("Chưa tìm thấy compiler C++. Hãy cài LLVM/Clang hoặc MinGW và thêm vào PATH.")
else:
    compiler_name = os.path.basename(compiler).lower()
    if compiler_name == "cl.exe":
        compile_command = [compiler, "/std:c++17", "/O2", "main.cpp", "/Fe:main.exe"]
    else:
        compile_command = [compiler, "-std=c++17", "-O3", "main.cpp", "-o", "main.exe" if os.name == "nt" else "main"]
    run_command = ["main.exe"] if os.name == "nt" else ["./main"]
    print(f"Compiler được chọn: {compiler}")

Chưa tìm thấy compiler C++. Hãy cài LLVM/Clang hoặc MinGW và thêm vào PATH.


## Tiếp theo: nhiệm vụ chính

In [24]:
# System prompt quy định vai trò, định dạng phản hồi và yêu cầu tương đương output.
system_prompt = """
Nhiệm vụ của bạn là chuyển mã Python thành mã C++ hiệu năng cao (high performance).
Chỉ trả lời bằng mã C++. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã C++ phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""  # Kết thúc chuỗi system prompt.

def user_prompt_for(python):  # Tạo prompt chi tiết cho từng đoạn mã Python cần chuyển sang C++.
    # Trả về f-string để chèn tự động thông tin hệ thống, lệnh compiler và mã nguồn.
    return f"""
Port (chuyển) mã Python này sang C++ với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.cpp rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã C++.
Mã Python cần port:

```python
{python}
```
"""  # Kết thúc và trả chuỗi user prompt.

In [25]:
def messages_for(python):  # Đóng gói prompt theo định dạng message mà Chat Completions yêu cầu.
    return [  # Trả về cuộc hội thoại gồm chỉ dẫn hệ thống và yêu cầu người dùng.
        {"role": "system", "content": system_prompt},  # Message hệ thống đặt quy tắc chuyển mã.
        {"role": "user", "content": user_prompt_for(python)}  # Message người dùng chứa mã Python cụ thể.
    ]  # Kết thúc danh sách messages.
 

In [26]:
def write_output(cpp):  # Ghi mã C++ do model sinh ra vào file nguồn.
    with open("main.cpp", "w", encoding="utf-8", newline="") as f:  # Luôn ghi UTF-8 để không lỗi ký tự tiếng Việt trên Windows.
        f.write(cpp)  # Ghi toàn bộ chuỗi mã C++ vào file.

In [27]:
def port(model, python):  # Gửi mã Python tới model đã chọn để chuyển sang C++.
    client = clients.get(model)  # Tra client phù hợp với tên model trong từ điển clients.
    if client is None:  # Bắt trường hợp model không có trong cấu hình.
        return f"Lỗi cấu hình: chưa có client cho model '{model}'."

    cloud_keys = {  # Các model cloud cần API key tương ứng.
        "gpt-5": openai_api_key,
        "claude-sonnet-4-5-20250929": anthropic_api_key,
        "grok-4": grok_api_key,
        "gemini-3.1-pro-preview": google_api_key,
        "openai/gpt-oss-120b": groq_api_key,
        "qwen/qwen3-coder-30b-a3b-instruct": openrouter_api_key,
    }
    if model in cloud_keys and not cloud_keys[model]:  # Không gọi API khi chưa cấu hình khóa.
        return f"Lỗi cấu hình: chưa thiết lập API key cho model '{model}'."

    try:  # Giữ lỗi API trong giao diện Gradio thay vì làm callback bị crash.
        request = {"model": model, "messages": messages_for(python)}
        if model == "gpt-5":  # Chỉ gửi reasoning_effort cho model OpenAI hỗ trợ tham số này.
            request["reasoning_effort"] = "high"
        response = client.chat.completions.create(**request)  # Gọi API sinh mã.
        reply = response.choices[0].message.content  # Lấy nội dung phản hồi của lựa chọn đầu tiên.
        if not reply:  # Một số provider có thể trả content=None.
            return "Lỗi model: API trả về phản hồi rỗng, không có mã C++ để ghi."
        reply = reply.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()  # Bỏ hàng rào Markdown.
        write_output(reply)  # Lưu mã vừa sinh vào main.cpp để chuẩn bị biên dịch.
        return reply  # Trả mã C++ cho ô kết quả trên giao diện.
    except Exception as error:  # Hiển thị lỗi provider/model/key theo cách dễ chẩn đoán.
        return f"Lỗi khi gọi model '{model}': {error}"

In [28]:
# Lưu chương trình xấp xỉ số pi dưới dạng chuỗi để nạp sẵn vào giao diện.
pi = """
import time  # Cung cấp đồng hồ để đo thời gian thực thi.

def calculate(iterations, param1, param2):  # Tính tổng chuỗi Leibniz biến đổi dùng để xấp xỉ pi.
    result = 1.0  # Khởi tạo tổng với số hạng đầu tiên của chuỗi.
    for i in range(1, iterations+1):  # Lặp từ 1 đến đúng số vòng iterations.
        j = i * param1 - param2  # Tạo mẫu số dạng 4i-1.
        result -= (1/j)  # Trừ nghịch đảo của mẫu số 4i-1 khỏi tổng.
        j = i * param1 + param2  # Tạo mẫu số dạng 4i+1.
        result += (1/j)  # Cộng nghịch đảo của mẫu số 4i+1 vào tổng.
    return result  # Trả giá trị tổng chuỗi trước khi nhân 4.

start_time = time.time()  # Ghi mốc thời gian ngay trước phép tính nặng.
result = calculate(200_000_000, 4, 1) * 4  # Chạy 200 triệu vòng và nhân 4 để xấp xỉ pi.
end_time = time.time()  # Ghi mốc thời gian ngay sau khi tính xong.

print(f"Kết quả (Result): {result:.12f}")  # In kết quả với 12 chữ số sau dấu thập phân.
print(f"Thời gian thực thi (Execution Time): {(end_time - start_time):.6f} giây")  # In thời lượng chạy với độ chính xác 6 chữ số thập phân.
"""  # Kết thúc chuỗi chương trình Python mẫu.

In [29]:
def run_python(code):  # Thực thi chuỗi mã Python và trả về nội dung mà mã đó in ra.
    globals_dict = {"__builtins__": __builtins__}  # Tạo không gian global riêng nhưng vẫn cho dùng hàm dựng sẵn.

    buffer = io.StringIO()  # Tạo bộ đệm trong RAM để hứng dữ liệu xuất chuẩn.
    old_stdout = sys.stdout  # Lưu stdout gốc để khôi phục sau khi chạy.
    sys.stdout = buffer  # Chuyển mọi print() tạm thời vào bộ đệm.

    try:  # Thử chạy mã và thu kết quả, đồng thời cho phép xử lý lỗi.
        exec(code, globals_dict)  # Thực thi chuỗi code trong không gian global riêng.
        output = buffer.getvalue()  # Đọc toàn bộ nội dung đã được print vào bộ đệm.
    except Exception as e:  # Bắt mọi lỗi Python phát sinh khi thực thi đoạn mã.
        output = f"Lỗi (Error): {e}"  # Chuyển lỗi thành chuỗi để hiển thị.
    finally:  # Khối này luôn chạy dù thành công hay lỗi.
        sys.stdout = old_stdout  # Khôi phục stdout để notebook tiếp tục in bình thường.

    return output  # Trả output hoặc thông báo lỗi về phía gọi hàm.

In [35]:
def compile_and_run():  # Biên dịch main.cpp rồi chạy chương trình ba lần để kiểm tra độ ổn định.
    if compile_command is None or run_command is None:  # Dừng sớm nếu máy chưa có compiler.
        print("Không thể biên dịch: chưa cài clang++, g++ hoặc MSVC cl.exe trong PATH.")
        print("Cài LLVM/Clang hoặc MinGW, thêm thư mục bin vào PATH, rồi khởi động lại kernel.")
        return

    try:  # Thử biên dịch và thực thi để có thể xử lý lỗi tiến trình.
        subprocess.run(compile_command, check=True, text=True, capture_output=True)  # Chạy compiler.
        for _ in range(3):  # Chạy ba lần để giảm ảnh hưởng sai lệch của một lần đo.
            result = subprocess.run(run_command, check=True, text=True, capture_output=True)
            print(result.stdout)
    except FileNotFoundError:
        print("Không tìm thấy compiler hoặc file thực thi. Hãy kiểm tra PATH và cài C++ compiler.")
    except subprocess.CalledProcessError as error:  # Bắt lỗi do compiler hoặc binary trả mã thoát khác 0.
        print(f"Đã xảy ra lỗi (error):\n{error.stderr or error.stdout}")

In [31]:
with gr.Blocks() as ui:  # Tạo ứng dụng Gradio và lưu cấu hình giao diện vào biến ui.
    with gr.Row():  # Xếp hai vùng nhập/xuất mã trên cùng một hàng.
        python = gr.Textbox(label="Mã Python:", lines=28, value=pi)  # Ô mã nguồn Python, nạp sẵn ví dụ tính pi.
        cpp = gr.Textbox(label="Mã C++:", lines=28)  # Ô hiển thị mã C++ do model sinh ra.
    with gr.Row():  # Tạo hàng điều khiển phía dưới hai ô mã.
        model = gr.Dropdown(models, label="Chọn model", value=models[0])  # Menu chọn model, mặc định phần tử đầu tiên.
        convert = gr.Button("Chuyển mã")  # Nút bắt đầu yêu cầu model chuyển mã.

    convert.click(port, inputs=[model, python], outputs=[cpp])  # Gắn sự kiện click: gọi port rồi đưa kết quả vào ô C++.

ui.launch(inbrowser=True)  # Khởi động server Gradio và tự mở giao diện trong trình duyệt.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [39]:
compile_and_run()  # Biên dịch main.cpp hiện có rồi chạy binary ba lần để quan sát kết quả/thời gian.

Không thể biên dịch: chưa cài clang++, g++ hoặc MSVC cl.exe trong PATH.
Cài LLVM/Clang hoặc MinGW, thêm thư mục bin vào PATH, rồi khởi động lại kernel.


Qwen 2.5 Coder: Fail (thất bại)  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383




Trong thí nghiệm của Ed, các mức tăng tốc (performance speedup) là:

Hạng 9: Qwen 2.5 Coder: Fail (thất bại)  
Hạng 8: OpenAI GPT-OSS 120B: 14X speedup (tăng tốc)    
Hạng 7: DeepSeek Coder v2: 168X speedup (tăng tốc)  
Hạng 6: Qwen3 Coder 30B: 168X speedup (tăng tốc)   
Hạng 5: Claude Sonnet 4.5: 184X speedup (tăng tốc)   
Hạng 4: GPT-5: 233X speedup (tăng tốc)  
**Hạng 3: oss-20B: 238X speedup (tăng tốc)**  
Hạng 2: Grok 4: 1060X speedup (tăng tốc)  
Hạng 1: Gemini 2.5 Pro: 1440X speedup (tăng tốc)  